In [5]:
import json
from pathlib import Path
import google.generativeai as genai
import os
from langchain_core.prompts import PromptTemplate
import time
from dotenv import load_dotenv

In [6]:
gemini_api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key = gemini_api_key)
model = genai.GenerativeModel('gemini-pro')

prompt = PromptTemplate.from_template("""
    Task: You are an AI assistant that generates QA pairs from a given text extracted from a book. Generate specific question-and-answer pairs from the given input text.

    Input Text:
    {input}

    Instruction: Return a JSON object in the response that strictly matches the following structure:

    [
        {{
            "Question": "Example question 1",
            "Answer": "Example answer 1"
        }},
        {{
            "Question": "Example question 2",
            "Answer": "Example answer 2"
        }}
        // ... additional question-answer pairs ...
    ]

    Output:
""")

adaptorPrompt = PromptTemplate.from_template("""
    Task: You are an AI assistant that converts given text into correct json format.

                                                
    Input Text: 
    {input}
                                                                                                                              
    Instruction: Return a JSON object in the response that strictly matches the following structure:

        [
            {{
                "Question": "Example question 1",
                "Answer": "Example answer 1"
            }},
            {{
                "Question": "Example question 2",
                "Answer": "Example answer 2"
            }}
            // ... additional question-answer pairs ...
        ]
""")


In [7]:
def generate_question_answer(chunk):
    formatted_prompt = prompt.format(input=chunk['input_text'])
    # print(f"{formatted_prompt}=")

    try:
        response = model.generate_content(formatted_prompt).text
        print(f"{response=}")
        formatted_adapted_prompt = adaptorPrompt.format(input=response)
        chunk['qa_pairs'] = json.loads(response)
    except json.JSONDecodeError:
            while True:
                try:
                    time.sleep(5)
                    response = model.generate_content(formatted_prompt).text
                    print(f"{response=}")
                    chunk['qa_pairs'] = json.loads(response)
                    break
                except json.JSONDecodeError:
                    continue
    return response 
    

In [8]:
chunks_path = Path('booksChunks')
dataset_path = Path('Dataset')

for file in chunks_path.glob("*.json"):
    with open(file, 'r') as json_file:
        chunks_dict = json.load(json_file)
    count=0
    for chunk in chunks_dict:
        # Generate QA pairs and parse them as JSON
        chunk = generate_question_answer(chunk)
        print(chunk)
        print(f"Chunk {count} done")
        count+=1
        # if count==5:
        #     break
        
    # Save the updated chunks back to the file
    with open(os.path.join(dataset_path, os.path.basename(file)), 'w') as json_file:
        json.dump(chunks_dict, json_file, indent=2)


response='```json\n[]\n```'
response='[\n  {\n    "Question": "Who are the authors of the book \\"Fundamentals of Deep Learning, Second Edition\\"?",\n    "Answer": "Nithin Buduma, Nikhil Buduma, Nicholas Locascio, and Joe Papa."\n  },\n  {\n    "Question": "What is the purpose of Early Release ebooks?",\n    "Answer": "To provide readers with access to the author\'s raw and unedited content as they write, allowing them to take advantage of the latest technologies before the books\' official release."\n  },\n  {\n    "Question": "Who holds the copyright for \\"Fundamentals of Deep Learning\\"?",\n    "Answer": "Nithin Buduma and Mobile Insights Technology Group, LLC."\n  },\n  {\n    "Question": "Where is O\'Reilly Media, Inc. located?",\n    "Answer": "1005 Gravenstein Highway North, Sebastopol, CA 95472."\n  }\n]'
[
  {
    "Question": "Who are the authors of the book \"Fundamentals of Deep Learning, Second Edition\"?",
    "Answer": "Nithin Buduma, Nikhil Buduma, Nicholas Locascio, 

ResourceExhausted: 429 Resource has been exhausted (e.g. check quota).